Abordagem não supervisionada: utilização do corpus financeiro completo para exploração de padrões, agrupamentos e identificação de possíveis observações atípicas.

# 🚀 Configuração do Ambiente

Este notebook faz parte do projeto **FinCheck**.

Para executar o notebook localmente, siga os passos abaixo.

---

## 1. Clonar o repositório

No terminal:

```bash
git clone https://github.com/SEU-USUARIO/FinCheck.git
```

## Entre na pasta do projeto:
cd FinCheck

## Crie o ambiente virtual
```bash
python -m venv .venv
```
O ambiente virtual não é versionado no GitHub. Cada usuário deve criar seu próprio ambiente localmente.

## Ative o ambiente virtual:
```bash
source .venv/Scripts/activate
```
Após a ativação, o terminal deverá apresentar algo semelhante a:
(.venv)

## Instalar as dependências
```bash
pip install -r requirements.txt
```

## Verificar as principais dependências
```bash
python -c "import pandas, numpy, sklearn, matplotlib, seaborn, nltk, wordcloud; print('Todas as dependencias principais OK!')"
```

## Verificar o Jupyter
```bash
jupyter --version
```

## Executar pelo VS Code

Caso esteja utilizando o VS Code:

Abra a pasta FinCheck.
Abra o arquivo .ipynb.
Selecione o Kernel/Interpretador Python.
Escolha o ambiente: (.venv) ou .venv/Scripts/python.exe

## Observação:
A pasta .venv/ não deve ser enviada para o GitHub.
O arquivo requirements.txt deve ser enviado para o GitHub.
Os notebooks .ipynb devem ser enviados para o GitHub.
Consulte o README.md para informações adicionais sobre os datasets utilizados pelo projeto.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sklearn
import nltk

from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN, KMeans, MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD
from nltk.corpus import stopwords


nltk.download("stopwords")


In [ ]:
url = "https://huggingface.co/datasets/lucasalmda/pt-br-financial-news-dataset/resolve/main/financial_news_br.jsonl"

finews = pd.read_json(url, lines=True)

print("Dataset carregado com sucesso!")
print("Dimensões:", finews.shape)

In [ ]:
display(finews.head())

In [ ]:
print("Colunas:")
print(finews.columns.tolist())

In [ ]:
print(finews["source"].value_counts())

In [ ]:
print(finews.isnull().sum())

In [ ]:
print("Total de notícias:", len(finews))
print("Com descrição:", finews["description"].notna().sum())
print("Sem descrição:", finews["description"].isna().sum())

In [ ]:
finews_limpo = finews.dropna(subset=["description"]).copy()

print("Dimensões após a limpeza:", finews_limpo.shape)
print("Descrições nulas:", finews_limpo["description"].isnull().sum())

In [ ]:
print("Títulos duplicados:", finews_limpo["title"].duplicated().sum())
print("URLs duplicadas:", finews_limpo["url"].duplicated().sum())
print(
    "Título + descrição duplicados:",
    finews_limpo.duplicated(subset=["title", "description"]).sum()
)

In [ ]:
finews_limpo = finews_limpo.drop_duplicates(
    subset=["title", "description"]
).copy()

print("Dimensões após remover duplicatas:", finews_limpo.shape)

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

print("Pasta processed verificada/criada.")

In [ ]:
caminho_saida = "../data/processed/financial_news_br_limpo.csv"

finews_limpo.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset salvo com sucesso!")
print("Caminho:", caminho_saida)
print("Dimensões:", finews_limpo.shape)

EDA -> Análise Exploratória de Dados

In [ ]:
finews_limpo["source"].value_counts().plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Distribuição das notícias por fonte")
plt.xlabel("Fonte")
plt.ylabel("Quantidade de notícias")
plt.xticks(rotation=0)

plt.show()

In [ ]:
finews_limpo["published_at"] = pd.to_datetime(
    finews_limpo["published_at"],
    errors="coerce"
)
finews_limpo["published_at"].describe()
finews_limpo["year"] = finews_limpo["published_at"].dt.year
finews_limpo["year"].value_counts().sort_index()

contagem = finews_limpo["year"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(contagem.index, contagem.values, color="steelblue")  # ou plt.plot(...)
plt.title("Distribuição das notícias por ano")
plt.xlabel("Ano")
plt.ylabel("Quantidade de notícias")
plt.xticks(contagem.index, rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VARIÁVEIS PARA A ANÁLISE TEXTUAL
# ============================================================

finews_limpo["published_at"] = pd.to_datetime(
    finews_limpo["published_at"],
    errors="coerce"
)

finews_limpo["year"] = finews_limpo["published_at"].dt.year

finews_limpo["title_chars"] = (
    finews_limpo["title"]
    .fillna("")
    .astype(str)
    .str.len()
)

finews_limpo["text_chars"] = (
    finews_limpo["sentiment_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

finews_limpo["title_words"] = (
    finews_limpo["title"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

finews_limpo["text_words"] = (
    finews_limpo["sentiment_text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

plt.figure(figsize=(10, 5))

plt.hist(
    finews_limpo["text_chars"],
    bins=50
)

plt.title("Distribuição do tamanho das notícias")
plt.xlabel("Quantidade de caracteres")
plt.ylabel("Quantidade de notícias")

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=finews_limpo,
    x="source",
    y="text_chars"
)

plt.title("Distribuição do tamanho das notícias por fonte")
plt.xlabel("Fonte")
plt.ylabel("Quantidade de caracteres")

plt.xticks(rotation=20)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.violinplot(
    data=finews_limpo,
    x="source",
    y="text_chars",
    inner="quartile"
)

plt.title("Distribuição da extensão das notícias por fonte")
plt.xlabel("Fonte")
plt.ylabel("Quantidade de caracteres")

plt.xticks(rotation=20)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    finews_limpo["text_words"],
    bins=50
)

plt.title("Distribuição da quantidade de palavras nas notícias")
plt.xlabel("Quantidade de palavras")
plt.ylabel("Quantidade de notícias")

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    finews_limpo["title_chars"],
    finews_limpo["text_chars"],
    alpha=0.05
)

plt.title("Relação entre tamanho do título e tamanho da notícia")
plt.xlabel("Caracteres no título")
plt.ylabel("Caracteres na notícia")

plt.show()

In [ ]:
finews_limpo["tags"].head(10)

In [ ]:
from collections import Counter

todas_tags = []

for tags in finews_limpo["tags"].dropna():

    if isinstance(tags, list):
        todas_tags.extend(tags)

contador_tags = Counter(todas_tags)

tags_df = pd.DataFrame(
    contador_tags.most_common(20),
    columns=["tag", "quantidade"]
)

display(tags_df)

In [ ]:
plt.figure(figsize=(10, 7))

plt.barh(
    tags_df["tag"][::-1],
    tags_df["quantidade"][::-1]
)

plt.title("20 tags mais frequentes")
plt.xlabel("Quantidade de ocorrências")
plt.ylabel("Tag")

plt.show()

In [ ]:
# Stopwords em português
stopwords_pt = set(stopwords.words("portuguese"))

# Junta título + descrição
texto = " ".join(
    finews_limpo["title"].fillna("") + " " +
    finews_limpo["description"].fillna("")
)

# Gera a nuvem
wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    stopwords=stopwords_pt,
    max_words=100
).generate(texto)

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Nuvem de palavras das notícias financeiras")
plt.show()

In [ ]:
tags = finews_limpo["tags"].value_counts()

print(tags.head(20))

In [ ]:
plt.figure(figsize=(10, 6))

tags.head(15).plot(
    kind="bar"
)

plt.title("15 tags mais frequentes")
plt.xlabel("Tag")
plt.ylabel("Quantidade de notícias")
plt.xticks(rotation=45, ha="right")

plt.show()

In [ ]:
print(
    "Duplicatas de título:",
    finews_limpo["title"].duplicated().sum()
)

print(
    "Duplicatas de URL:",
    finews_limpo["url"].duplicated().sum()
)

print(
    "Duplicatas de título + descrição:",
    finews_limpo.duplicated(
        subset=["title", "description"]
    ).sum()
)

In [ ]:
finews_limpo[
    ["title_chars", "text_chars"]
].describe()

In [ ]:
finews_limpo.describe()

In [ ]:
finews_limpo.describe(include="all")

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    finews_limpo["title_chars"],
    bins=40
)

plt.title("Distribuição do tamanho dos títulos")
plt.xlabel("Quantidade de caracteres")
plt.ylabel("Quantidade de notícias")

plt.show()

In [ ]:
correlacao = finews_limpo[
    ["title_chars", "text_chars"]
].corr()

print(correlacao)

In [ ]:
finews_limpo.columns.tolist()

In [ ]:
finews_limpo["text_clean"] = (
    finews_limpo["title"].fillna("") + " " +
    finews_limpo["description"].fillna("")
)

In [ ]:
finews_limpo[["title", "description", "text_clean"]].head()

TF-IDF

In [ ]:
# ============================================================
# TF-IDF
# ============================================================

tfidf_unsup = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words=list(stopwords_pt),
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    max_features=50000,
    sublinear_tf=True
)

X_tfidf = tfidf_unsup.fit_transform(
    finews_limpo["text_clean"]
)

print("Dimensão da matriz TF-IDF:")
print(X_tfidf.shape)

In [ ]:
indice_noticia = 0

print("Título:")
print(finews_limpo.iloc[indice_noticia]["title"])

scores_noticia = X_tfidf[indice_noticia].toarray().ravel()

termos = tfidf_unsup.get_feature_names_out()

tfidf_noticia = pd.DataFrame({
    "termo": termos,
    "score_tfidf": scores_noticia
})

tfidf_noticia = tfidf_noticia[
    tfidf_noticia["score_tfidf"] > 0
]

top_noticia = (
    tfidf_noticia
    .sort_values("score_tfidf", ascending=False)
    .head(20)
    .sort_values("score_tfidf")
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    top_noticia[["score_tfidf"]].T,
    cmap="YlGnBu",
    annot=True,
    fmt=".3f",
    cbar_kws={"label": "Score TF-IDF"}
)

plt.title("Top 20 termos por score TF-IDF — uma notícia")
plt.xlabel("Termos")
plt.ylabel("Notícia")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

REDUÇÃO TF-IDF

In [ ]:
svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

X_reduced = svd.fit_transform(X_tfidf)

print("Dimensão após redução:")
print(X_reduced.shape)

In [ ]:
# 20 primeiras notícias
X_amostra = X_tfidf[:20]

# Converte para DataFrame
tfidf_amostra = pd.DataFrame(
    X_amostra.toarray(),
    columns=tfidf_unsup.get_feature_names_out()
)

# 20 termos com maior score na amostra
top_termos = (
    tfidf_amostra
    .sum(axis=0)
    .sort_values(ascending=False)
    .head(20)
    .index
)

heatmap_data = tfidf_amostra[top_termos]

plt.figure(figsize=(14, 8))

sns.heatmap(
    heatmap_data,
    cmap="YlGnBu",
    linewidths=0.3
)

plt.title("Mapa de calor dos scores TF-IDF")
plt.xlabel("Termos")
plt.ylabel("Notícias")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Treinamento Não Supervisionado
kmeans

In [ ]:
# Número de clusters
K_FINAL = 5

# Criando o modelo K-Means
kmeans = KMeans(
    n_clusters=K_FINAL,
    random_state=42,
    n_init=10
)

# Treinando o modelo e atribuindo os clusters
finews_limpo["cluster"] = kmeans.fit_predict(X_reduced)

# Distribuição das notícias por cluster
print("Distribuição dos clusters:")
print(
    finews_limpo["cluster"]
    .value_counts()
    .sort_index()
)

In [ ]:
plt.figure(figsize=(10, 6))

contagem_kmeans = (
    finews_limpo["cluster"]
    .value_counts()
    .sort_index()
)

sns.barplot(
    x=contagem_kmeans.index,
    y=contagem_kmeans.values
)

plt.title("Distribuição das Notícias por Cluster — K-Means")
plt.xlabel("Cluster")
plt.ylabel("Quantidade de Notícias")

plt.tight_layout()
plt.show()

In [ ]:
# Avaliação do agrupamento
silhouette = silhouette_score(
    X_reduced,
    finews_limpo["cluster"]
)

print(f"\nSilhouette Score: {silhouette:.4f}")

In [ ]:
silhouette_comparacao = pd.DataFrame({
    "Modelo": ["K-Means", "MiniBatchKMeans"],
    "Silhouette": [0.0475, 0.0420]
})

plt.figure(figsize=(8, 6))

sns.barplot(
    data=silhouette_comparacao,
    x="Modelo",
    y="Silhouette"
)

plt.title("Comparação do Silhouette Score")
plt.xlabel("Modelo")
plt.ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

In [ ]:
Principais termos associados a cada cluster

termos = tfidf_unsup.get_feature_names_out()

for cluster in range(K_FINAL):

    noticias_cluster = finews_limpo["cluster"] == cluster

    # Média dos valores TF-IDF das notícias pertencentes ao cluster
    tfidf_cluster = X_tfidf[noticias_cluster.values]

    medias = tfidf_cluster.mean(axis=0).A1

    indices = medias.argsort()[::-1][:15]

    print(f"\nCluster {cluster}")
    print("-" * 50)
    print(", ".join(termos[i] for i in indices))

In [ ]:
# ============================================================
# REDUÇÃO PARA VISUALIZAÇÃO EM 2D
# ============================================================

svd_2d = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd_2d.fit_transform(X_tfidf)

print("Dimensão após redução:")
print(X_2d.shape)

In [ ]:
# ============================================================
# VISUALIZAÇÃO DOS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(
    x=X_2d[:, 0],
    y=X_2d[:, 1],
    hue=finews_limpo["cluster"],
    palette="tab10",
    s=20,
    alpha=0.5,
    legend="full"
)

plt.title("Clusters das Notícias Financeiras — K-Means")
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

MiniBatchKMeans

In [ ]:
K_FINAL = 5

mbkmeans = MiniBatchKMeans(
    n_clusters=K_FINAL,
    random_state=42,
    batch_size=256,
    n_init=10
)

finews_limpo["cluster_mini"] = mbkmeans.fit_predict(
    X_reduced
)

print("Distribuição dos clusters — MiniBatchKMeans:")

print(
    finews_limpo["cluster_mini"]
    .value_counts()
    .sort_index()
)

In [ ]:
silhouette_mini = silhouette_score(
    X_reduced,
    finews_limpo["cluster_mini"]
)

print(f"Silhouette Score — MiniBatchKMeans: {silhouette_mini:.4f}")

In [ ]:
silhouette_kmeans = silhouette_score(
    X_reduced,
    finews_limpo["cluster"]
)

print(f"K-Means:        {silhouette_kmeans:.4f}")
print(f"MiniBatchKMeans: {silhouette_mini:.4f}")

In [ ]:
comparacao_clusters = pd.DataFrame({
    "K-Means": finews_limpo["cluster"].value_counts().sort_index(),
    "MiniBatchKMeans": finews_limpo["cluster_mini"].value_counts().sort_index()
})

print(comparacao_clusters)

In [ ]:
comparacao_clusters_plot = comparacao_clusters.reset_index()

comparacao_clusters_plot = comparacao_clusters_plot.rename(
    columns={"index": "Cluster"}
)

comparacao_clusters_plot = comparacao_clusters_plot.melt(
    id_vars="Cluster",
    var_name="Modelo",
    value_name="Quantidade"
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=comparacao_clusters_plot,
    x="Cluster",
    y="Quantidade",
    hue="Modelo"
)

plt.title("Distribuição dos Clusters — K-Means vs MiniBatchKMeans")
plt.xlabel("Rótulo do Cluster")
plt.ylabel("Quantidade de Notícias")

plt.tight_layout()
plt.show()

DBSCAN

In [ ]:
eps_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

resultados_dbscan = []

for eps in eps_values:

    dbscan = DBSCAN(
        eps=eps,
        min_samples=10
    )

    labels = dbscan.fit_predict(X_reduced)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()

    resultados_dbscan.append({
        "eps": eps,
        "clusters": n_clusters,
        "ruido": n_noise,
        "percentual_ruido": n_noise / len(labels) * 100
    })

resultados_dbscan = pd.DataFrame(resultados_dbscan)

print(resultados_dbscan)

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=resultados_dbscan,
    x="eps",
    y="clusters",
    marker="o"
)

plt.title("DBSCAN — Número de Clusters por Valor de EPS")
plt.xlabel("EPS")
plt.ylabel("Número de Clusters")

plt.xticks(resultados_dbscan["eps"])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=resultados_dbscan,
    x="eps",
    y="percentual_ruido",
    marker="o"
)

plt.title("DBSCAN — Percentual de Notícias Classificadas como Ruído")
plt.xlabel("EPS")
plt.ylabel("Ruído (%)")

plt.xticks(resultados_dbscan["eps"])

plt.tight_layout()
plt.show()

In [ ]:
dbscan_final = DBSCAN(
    eps=0.3,
    min_samples=10
)

finews_limpo["cluster_dbscan"] = dbscan_final.fit_predict(
    X_reduced
)

print("Distribuição dos clusters — DBSCAN:")
print(
    finews_limpo["cluster_dbscan"]
    .value_counts()
    .sort_index()
)

In [ ]:
contagem_dbscan = (
    finews_limpo["cluster_dbscan"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(12, 6))

sns.barplot(
    x=contagem_dbscan.index.astype(str),
    y=contagem_dbscan.values
)

plt.title("Distribuição das Notícias — DBSCAN (EPS = 0.3)")
plt.xlabel("Cluster")
plt.ylabel("Quantidade de Notícias")

plt.tight_layout()
plt.show()

In [ ]:
ruido_dbscan = (finews_limpo["cluster_dbscan"] == -1).sum()
total_noticias = len(finews_limpo)

percentual_ruido = ruido_dbscan / total_noticias * 100

print(f"Notícias classificadas como ruído: {ruido_dbscan}")
print(f"Total de notícias: {total_noticias}")
print(f"Percentual de ruído: {percentual_ruido:.2f}%")

In [ ]:
plt.figure(figsize=(12, 8))

sns.scatterplot(
    x=X_2d[:, 0],
    y=X_2d[:, 1],
    hue=finews_limpo["cluster_dbscan"],
    palette="tab10",
    s=20,
    alpha=0.5,
    legend="full"
)

plt.title("Clusters das Notícias Financeiras — DBSCAN")
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

Gaussian Mixture

In [ ]:
resultados_gmm = []

for n in range(2, 11):

    gmm = GaussianMixture(
        n_components=n,
        random_state=42
    )

    gmm.fit(X_reduced)

    resultados_gmm.append({
        "componentes": n,
        "AIC": gmm.aic(X_reduced),
        "BIC": gmm.bic(X_reduced)
    })

resultados_gmm = pd.DataFrame(resultados_gmm)

print(resultados_gmm)

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=resultados_gmm,
    x="componentes",
    y="AIC",
    marker="o",
    label="AIC"
)

sns.lineplot(
    data=resultados_gmm,
    x="componentes",
    y="BIC",
    marker="o",
    label="BIC"
)

plt.title("Gaussian Mixture — AIC e BIC")
plt.xlabel("Número de Componentes")
plt.ylabel("Critério de Informação")

plt.xticks(resultados_gmm["componentes"])
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
resultados_gmm = []

for n in range(2, 13):

    print(f"Testando {n} componentes...")

    gmm = GaussianMixture(
        n_components=n,
        covariance_type="diag",
        n_init=2,
        max_iter=300,
        random_state=42
    )

    gmm.fit(X_reduced)

    resultados_gmm.append({
        "componentes": n,
        "AIC": gmm.aic(X_reduced),
        "BIC": gmm.bic(X_reduced),
        "convergiu": gmm.converged_
    })

resultados_gmm = pd.DataFrame(resultados_gmm)

print(resultados_gmm)

Isolation Forest

In [ ]:
isolation_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

labels_if = isolation_forest.fit_predict(X_reduced)

finews_limpo["anomaly_if"] = labels_if

In [ ]:
print(
    finews_limpo["anomaly_if"]
    .value_counts()
    .sort_index()
)

In [ ]:
finews_limpo["tipo_anomalia"] = finews_limpo["anomaly_if"].map({
    1: "Normal",
    -1: "Anomalia"
})

In [ ]:
print(
    finews_limpo["tipo_anomalia"]
    .value_counts()
)

In [ ]:
quantidade_anomalias = (
    finews_limpo["anomaly_if"] == -1
).sum()

total_noticias = len(finews_limpo)

percentual_anomalias = (
    quantidade_anomalias / total_noticias
) * 100

print(f"Total de notícias: {total_noticias}")
print(f"Anomalias: {quantidade_anomalias}")
print(f"Percentual de anomalias: {percentual_anomalias:.2f}%")

In [ ]:
plt.figure(figsize=(8, 6))

contagem_anomalias = (
    finews_limpo["tipo_anomalia"]
    .value_counts()
)

sns.barplot(
    x=contagem_anomalias.index,
    y=contagem_anomalias.values
)

plt.title("Isolation Forest — Notícias Normais e Anômalas")
plt.xlabel("Classificação")
plt.ylabel("Quantidade de Notícias")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

sns.scatterplot(
    x=X_2d[:, 0],
    y=X_2d[:, 1],
    hue=finews_limpo["tipo_anomalia"],
    s=20,
    alpha=0.5
)

plt.title("Isolation Forest — Visualização das Anomalias")
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")

plt.legend(title="Classificação")

plt.tight_layout()
plt.show()

In [ ]:
scores_if = isolation_forest.decision_function(X_reduced)

finews_limpo["score_anomalia"] = scores_if

In [ ]:
print(
    finews_limpo[
        ["title", "score_anomalia", "tipo_anomalia"]
    ].head(10)
)

In [ ]:
noticias_anomalas = (
    finews_limpo[
        finews_limpo["anomaly_if"] == -1
    ]
    .sort_values("score_anomalia")
)

print(
    noticias_anomalas[
        ["title", "source", "published_at", "score_anomalia"]
    ].head(20)
)

In [ ]:
for _, noticia in noticias_anomalas.head(10).iterrows():
    print("=" * 80)
    print("Título:", noticia["title"])
    print("Fonte:", noticia["source"])
    print("Data:", noticia["published_at"])
    print("Score:", noticia["score_anomalia"])

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    finews_limpo["score_anomalia"],
    bins=50,
    kde=True
)

plt.title("Distribuição dos Scores — Isolation Forest")
plt.xlabel("Score de Anomalia")
plt.ylabel("Quantidade de Notícias")

plt.tight_layout()
plt.show()

In [ ]:
print(finews_limpo["anomaly_if"].value_counts().sort_index())

In [ ]:
print(
    finews_limpo[
        ["title", "source", "published_at", "score_anomalia"]
    ]
    .sort_values("score_anomalia")
    .head(20)
)

In [ ]:
anomalias = finews_limpo[
    finews_limpo["anomaly_if"] == -1
]

print(
    anomalias["source"]
    .value_counts()
)

In [ ]:
print(
    anomalias["year"]
    .value_counts()
    .sort_index()
)

In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=anomalias,
    x="year"
)

plt.title("Notícias Anômalas por Ano — Isolation Forest")
plt.xlabel("Ano")
plt.ylabel("Quantidade de Anomalias")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
top_anomalias = (
    finews_limpo[
        finews_limpo["anomaly_if"] == -1
    ]
    .sort_values("score_anomalia")
    .head(20)
)

top_anomalias[
    [
        "title",
        "source",
        "published_at",
        "score_anomalia"
    ]
]

In [ ]:
for _, noticia in top_anomalias.iterrows():

    print("=" * 100)

    print("Título:")
    print(noticia["title"])

    print("\nDescrição:")
    print(noticia["description"])

    print("\nFonte:", noticia["source"])
    print("Data:", noticia["published_at"])
    print("Score:", noticia["score_anomalia"])

In [ ]:
contagem_if = (
    finews_limpo["tipo_anomalia"]
    .value_counts()
    .reindex(["Normal", "Anomalia"])
)

plt.figure(figsize=(8, 6))

sns.barplot(
    x=contagem_if.index,
    y=contagem_if.values
)

plt.title("Isolation Forest — Notícias Normais e Anômalas")
plt.xlabel("Classificação")
plt.ylabel("Quantidade de Notícias")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=finews_limpo,
    x="score_anomalia",
    bins=50,
    kde=True
)

plt.title("Distribuição dos Scores — Isolation Forest")
plt.xlabel("Score de Anomalia")
plt.ylabel("Quantidade de Notícias")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

sns.scatterplot(
    x=X_2d[:, 0],
    y=X_2d[:, 1],
    hue=finews_limpo["tipo_anomalia"],
    s=20,
    alpha=0.5
)

plt.title("Isolation Forest — Visualização das Anomalias")
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")

plt.legend(title="Classificação")

plt.tight_layout()
plt.show()